In [8]:
# Step 1: Install tiktoken and import libraries
!pip install tiktoken pandas torch

import pandas as pd
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader


In [9]:
# Step 2: Set seed and device
torch.manual_seed(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)


Using: cuda


In [10]:
# Step 3: Load dataset
# Adjust filename if necessary; standard IMDB CSVs contain 'review' and 'sentiment' columns
df = pd.read_csv('IMDB Dataset.csv')

# Convert text sentiments ('positive'/'negative') to binary integers (1/0)
if df['sentiment'].dtype == object:
    df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

texts = df['review'].tolist()
labels = df['sentiment'].tolist()


In [11]:
# Step 4: Tokenization using tiktoken (cl100k_base encoding)
tokenizer = tiktoken.get_encoding("cl100k_base")

def encode_and_pad(text, max_len=128):
    tokens = tokenizer.encode(text)
    # Truncate if token length exceeds max_len
    if len(tokens) > max_len:
        tokens = tokens[:max_len]
    # Pad with zeros if token length is shorter than max_len
    else:
        tokens = tokens + [0] * (max_len - len(tokens))
    return tokens

# Convert all reviews to fixed-length token ID sequences
X_encoded = [encode_and_pad(text, max_len=128) for text in texts]


In [12]:
# Step 5: Convert token sequences and labels into PyTorch tensors
X = torch.tensor(X_encoded, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32)


In [13]:
# Step 6: Create PyTorch DataLoader for batch handling
class IMDBDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = IMDBDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)



In [14]:
# Step 7: Define the RNN Model (Kept structural match to your base code)
class MovieRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64)
        self.rnn = nn.RNN(64, 32, batch_first=True)
        self.output = nn.Linear(32, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.rnn(x)
        return self.output(hidden[-1]).squeeze(1)

# tiktoken max token vocabulary size for cl100k_base is 100,256
vocab_size = tokenizer.n_vocab
movie = MovieRNN(vocab_size).to(device)



In [15]:
# Step 8: Loss Function & Optimizer
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(movie.parameters(), lr=0.001)



In [16]:
# Step 9: Model Training Loop
epochs = 10
movie.train()

for epoch in range(epochs):
    total_loss = 0.0
    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        logits = movie(batch_X)
        loss = loss_fn(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs}: Loss = {total_loss / len(loader):.4f}")



Epoch 1/10: Loss = 0.6949
Epoch 2/10: Loss = 0.6839
Epoch 3/10: Loss = 0.6769
Epoch 4/10: Loss = 0.6704
Epoch 5/10: Loss = 0.6467
Epoch 6/10: Loss = 0.6292
Epoch 7/10: Loss = 0.6372
Epoch 8/10: Loss = 0.5859
Epoch 9/10: Loss = 0.5724
Epoch 10/10: Loss = 0.5435


In [17]:
# Step 10: Evaluation / Inference Example
movie.eval()
test_review = "This movie was absolutely amazing and wonderful!"

with torch.no_grad():
    tokens = torch.tensor([encode_and_pad(test_review, max_len=128)]).to(device)
    probability = torch.sigmoid(movie(tokens)).item()
    sentiment = "positive" if probability >= 0.5 else "negative"
    print(f"\nTest Review: '{test_review}'")
    print(f"Predicted Sentiment: {sentiment} (Probability: {probability:.4f})")


Test Review: 'This movie was absolutely amazing and wonderful!'
Predicted Sentiment: positive (Probability: 0.8665)


In [18]:
# Function to evaluate custom review strings
def predict_sentiment(review_text):
    movie.eval()  # Set model to evaluation mode
    with torch.no_grad():
        # Tokenize, truncate, and pad the input string
        tokens = torch.tensor([encode_and_pad(review_text, max_len=128)]).to(device)

        # Forward pass through the model
        logit = movie(tokens)
        probability = torch.sigmoid(logit).item()

        sentiment = "Positive" if probability >= 0.5 else "Negative"
        print(f"Review: '{review_text}'")
        print(f"Prediction: {sentiment} | Probability: {probability:.4f}\n")

# --- TEST EXAMPLES ---
predict_sentiment("This was the worst movie I have ever seen. Complete waste of time.")
predict_sentiment("An absolute masterpiece with incredible acting and directed brilliantly.")
predict_sentiment("It was okay, nothing special but had a few decent scenes.")
predict_sentiment("The movie was bad.")

Review: 'This was the worst movie I have ever seen. Complete waste of time.'
Prediction: Negative | Probability: 0.4584

Review: 'An absolute masterpiece with incredible acting and directed brilliantly.'
Prediction: Positive | Probability: 0.8665

Review: 'It was okay, nothing special but had a few decent scenes.'
Prediction: Negative | Probability: 0.4584

Review: 'The movie was bad.'
Prediction: Negative | Probability: 0.4584

